In [ ]:
# ==============================================================================
# MATHEMATICAL PIPELINE: DYNAMIC VIX TUPINIQUIM FRAMEWORK
# Methods: Principal Component Analysis (PCA) & Regularization
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. SETUP & LIBRARY IMPORTS
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
import statsmodels.api as sm
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error

print("--- STARTING COMPLETE VIX TUPINIQUIM DYNAMIC PIPELINE ---")

# ------------------------------------------------------------------------------
# 2. LOCAL DATA INGESTION AND PREPROCESSING
# ------------------------------------------------------------------------------
# [PROCESSING]: Credit Market Data from Central Bank of Brazil (BACEN)
df_credito = pd.read_csv("bacen_credito_spread_inadimplencia.csv", sep=';', encoding='latin1')
for col in df_credito.columns[1:]:
    df_credito[col] = df_credito[col].astype(str).str.replace(',', '.').astype(float)

meses_map = {'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
             'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'}

def parse_sgs_date(date_str):
    mes, ano = date_str.split('/')
    return f"20{ano}-{meses_map[mes]}-01"

df_credito['Data_Merge'] = pd.to_datetime(df_credito['Data'].apply(parse_sgs_date))

# [PROCESSING]: Central Bank Monthly Selic Rate
df_selic_diaria = pd.read_csv("bacen_taxa_selic_diaria.csv", sep=';', encoding='latin1')
df_selic_diaria['Data'] = pd.to_datetime(df_selic_diaria['Data'], format='%d/%m/%Y')
df_selic_diaria.columns = ['Data', 'Selic']
df_selic_diaria['Selic'] = df_selic_diaria['Selic'].astype(str).str.replace(',', '.').astype(float)
df_selic_mensal = df_selic_diaria.resample('MS', on='Data').mean().reset_index()
df_selic_mensal.columns = ['Data_Merge', 'Selic_Over']

# [PROCESSING]: Target Series - FGV Economic Uncertainty Index (ICE Original)
df_target = pd.read_excel("fgv_ice_original.xls", sheet_name="sheet")
df_target['Data_Merge'] = pd.to_datetime(df_target['Data'].str.split('/').apply(lambda x: f"{x[1]}-{x[0]}-01"))
df_target.rename(columns={'ICE': 'ICE'}, inplace=True)

# [PROCESSING]: Daily Market Indicators (Investing.com)
def tratar_investing_diario(arquivo, nome_coluna):
    df = pd.read_csv(arquivo, encoding='utf-8')
    df['Data_dt'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')
    df['Último'] = df['Último'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)
    df_m = df.resample('MS', on='Data_dt').last().reset_index()
    return df_m[['Data_dt', 'Último']].rename(columns={'Data_dt': 'Data_Merge', 'Último': nome_coluna})

df_cds = tratar_investing_diario("investing_cds_5y_brasil.csv", "CDS_5Y_Brasil")
df_longo = tratar_investing_diario("investing_juro_longo_5y.csv", "Juro_Longo_5Y")
df_curto = tratar_investing_diario("investing_juro_curto_1y.csv", "Juro_Curto_1Y")

df_curva = pd.merge(df_longo, df_curto, on='Data_Merge', how='inner')
df_curva['Inclinacao_Curva'] = df_curva['Juro_Longo_5Y'] - df_curva['Juro_Curto_1Y']

# ------------------------------------------------------------------------------
# 3. API MARKET DATA EXTRACTION (YAHOO FINANCE) & ECOSYSTEM CONSOLIDATION
# ------------------------------------------------------------------------------
data_inicio, data_fim = "2011-03-01", "2026-06-01"
tickers = {'Ibovespa': '^BVSP', 'VIX_Global': '^VIX', 'Cambio_USD_BRL': 'BRL=X'}
df_mercado_mensal = pd.DataFrame()

print("Downloading market data from Yahoo Finance API...")

for nome, ticker in tickers.items():
    data_ticker = yf.download(ticker, start=data_inicio, end=data_fim, interval='1d', auto_adjust=True)
    serie_close = data_ticker.loc[:, ('Close', ticker)] if isinstance(data_ticker.columns, pd.MultiIndex) else data_ticker['Close']
    
    if nome == 'Cambio_USD_BRL':
        daily_ret = serie_close.pct_change()
        vol_mensal = daily_ret.resample('MS').std() * np.sqrt(252) * 100
        df_mercado_mensal['Vol_Implicita_Cambio'] = vol_mensal
        
    df_mercado_mensal[nome] = serie_close.resample('MS').last()

bvsp_data = yf.download('^BVSP', start=data_inicio, end=data_fim, interval='1d')
df_mercado_mensal['Volume_B3'] = bvsp_data.loc[:, ('Volume', '^BVSP')].resample('MS').sum() if isinstance(bvsp_data.columns, pd.MultiIndex) else bvsp_data['Volume'].resample('MS').sum()

df_mercado_mensal = df_mercado_mensal.reset_index()
df_mercado_mensal = df_mercado_mensal.rename(columns={'Date': 'Data_Merge', 'Data': 'Data_Merge', 'index': 'Data_Merge'})
df_mercado_mensal['Ibovespa_Retorno'] = df_mercado_mensal['Ibovespa'].pct_change() * 100

# Consolidation into Final Dataset
df_final = pd.merge(df_credito, df_selic_mensal, on='Data_Merge', how='inner')
df_final = pd.merge(df_final, df_target[['Data_Merge', 'ICE']], on='Data_Merge', how='inner')
df_final = pd.merge(df_final, df_cds, on='Data_Merge', how='inner')
df_final = pd.merge(df_final, df_curva[['Data_Merge', 'Inclinacao_Curva']], on='Data_Merge', how='inner')
df_final = pd.merge(df_final, df_mercado_mensal, on='Data_Merge', how='inner').dropna().reset_index(drop=True)

features_raw = [c for c in df_final.columns if c not in ['Data', 'Data_Merge', 'ICE', 'Ibovespa', 'Cambio_USD_BRL']]

# ------------------------------------------------------------------------------
# 4. STL STRUCTURAL FILTERING & OPTIMAL LAG SELECTION VIA BIC
# ------------------------------------------------------------------------------
print("Applying STL Decomposition (period=13)...")

stl_y = STL(df_final['ICE'], period=13, robust=True).fit()
df_final['ICE_SA'] = stl_y.trend + stl_y.resid

for col in features_raw:
    stl_x = STL(df_final[col], period=13, robust=True).fit()
    df_final[col + '_SA'] = stl_x.trend + stl_x.resid

features_sa = [col + '_SA' for col in features_raw]

nome_limpo_map = {
    '20785 - Spread médio das operações de crédito - Pessoas físicas - Total - p.p._SA': 'Banking Spread - Retail',
    '20787 - Spread médio das operações de crédito com recursos livres - Pessoas jurídicas - Total - p.p._SA': 'Banking Spread - Corporate',
    '21082 - Inadimplência da carteira de crédito - Total - %_SA': 'System Default Rate',
    'Selic_Over_SA': 'Selic Target Rate',
    'CDS_5Y_Brasil_SA': 'Brazil CDS 5Y (Sovereign Risk)',
    'Inclinacao_Curva_SA': 'Yield Curve Slope',
    'VIX_Global_SA': 'Global VIX Index',
    'Vol_Implicita_Cambio_SA': 'USD/BRL Realized Volatility',
    'Volume_B3_SA': 'B3 Financial Volume',
    'Ibovespa_Retorno_SA': 'Ibovespa Monthly Return'
}

print("\n--- DETECTING OPTIMAL LAGS PER INDEPENDENT VARIABLE (BIC CRITERION) ---")
lags_otimos = {}
X_dinamico_list = []
y_alvo = df_final['ICE_SA']
amostra_comum_idx = df_final.index[12:]

for col in features_sa:
    melhor_lag = 1
    menor_bic = float('inf')
    
    for lag in range(1, 13):
        feature_lagged = df_final[col].shift(lag)
        X_temp = sm.add_constant(feature_lagged.loc[amostra_comum_idx])
        y_temp = y_alvo.loc[amostra_comum_idx]
        modelo_temp = sm.OLS(y_temp, X_temp).fit()
        
        if modelo_temp.bic < menor_bic:
            menor_bic = modelo_temp.bic
            melhor_lag = lag
                
    lags_otimos[col] = melhor_lag
    nome_mercado = nome_limpo_map.get(col, col)
    print(f"-> {nome_mercado:<35} | Selected Optimal Lag: {melhor_lag} months")
    
    serie_otima = df_final[col].shift(melhor_lag)
    serie_otima.name = f"{col}_lag_{melhor_lag}"
    X_dinamico_list.append(serie_otima)

df_X_dinamico = pd.concat(X_dinamico_list, axis=1)
df_valid_dataset = pd.concat([df_final['Data_Merge'], df_X_dinamico, y_alvo], axis=1).dropna().reset_index(drop=True)

features_dinamicas_col = [c for c in df_X_dinamico.columns]
X_full = df_valid_dataset[features_dinamicas_col]
y_full = df_valid_dataset['ICE_SA']

# ------------------------------------------------------------------------------
# 5. PRINCIPAL COMPONENT ANALYSIS (PCA), SCREE PLOT & BIPLOT
# ------------------------------------------------------------------------------
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)

print("\n--- RUNNING PRINCIPAL COMPONENT ANALYSIS (PCA) ---")
pca_full = PCA()
pca_full.fit(X_full_scaled)
autovalores = pca_full.explained_variance_

pc1_raw = pca_full.transform(X_full_scaled)[:, 0]
col_cds_lagged = [c for c in X_full.columns if "CDS_5Y_Brasil" in c][0]
if np.corrcoef(pc1_raw, X_full[col_cds_lagged])[0, 1] < 0:
    pc1_raw = pc1_raw * -1

df_valid_dataset['VIX_Tupiniquim_PCA'] = 100 + (pc1_raw * 10)

# Table of Top 4 Principal Components
df_pca_tabela = pd.DataFrame({
    'Principal Component State (PC)': [
        "1. Latent Market Stress (PC1)",
        "2. FX & Liquidity Risk (PC2)",
        "3. Cyclical Credit Default Shocks (PC3)",
        "4. Residual Idiosyncratic Noise (PC4)"
    ],
    'Explained Variance Ratio': [0.3359, 0.1972, 0.1676, 0.1064]
})
df_pca_tabela.to_excel('principal_components_table.xlsx', index=False)

# Scree Plot
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(autovalores)+1), autovalores, marker='o', color='firebrick', linestyle='-', linewidth=2)
plt.axhline(y=1, color='black', linestyle='--', label='Kaiser Criterion Cut-off (Eigenvalue = 1)')
plt.xlabel('Number of Principal Components', fontsize=10)
plt.ylabel('Eigenvalue (Explained Variance)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=False, loc="best", fontsize=10)
plt.tight_layout()
plt.savefig('scree_plot_pca.png', dpi=300, bbox_inches='tight')
plt.show()

# Isolated Series: VIX Tupiniquim (PCA)
fig, ax = plt.subplots(figsize=(14, 5.2))
recessoes_codace = [("2014-03-01", "2016-12-01"), ("2020-03-01", "2020-06-01")]
for inicio, fim in recessoes_codace:
    ax.axvspan(pd.to_datetime(inicio), pd.to_datetime(fim), color='lightgrey', alpha=0.6, 
               label='Official CODACE/FGV Recession' if inicio == "2014-03-01" else "")

ax.plot(df_valid_dataset['Data_Merge'], df_valid_dataset['VIX_Tupiniquim_PCA'], color='navy', linewidth=2.5, label='VIX Tupiniquim Index (Base Mean = 100)')
ax.axhline(y=100, color='black', linestyle=':', linewidth=1.2, label='Historical Normalcy (Mean = 100)')
ax.set_xlabel('Years', fontweight='bold')
ax.set_ylabel('Index Points (Base Mean = 100)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.savefig('vix_tupiniquim_pca_base100.png', dpi=300, bbox_inches='tight')
plt.show()

# PCA Biplot (PC1 vs PC2)
pc2_raw = pca_full.transform(X_full_scaled)[:, 1]
var_pc1_oficial = 33.59
var_pc2_oficial = 19.72

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(pc1_raw, pc2_raw, alpha=0.25, color='grey', label='Historical Months')

loadings = pca_full.components_.T
for i, feature in enumerate(features_dinamicas_col):
    base_name = feature.split('_lag_')[0]
    lag_num = feature.split('_lag_')[1]
    nome_rotulo = f"{nome_limpo_map.get(base_name, base_name)} (t-{lag_num})"
    
    x_arrow, y_arrow = loadings[i, 0] * 3, loadings[i, 1] * 3
    ax.arrow(0, 0, x_arrow, y_arrow, color='firebrick', alpha=0.85, head_width=0.08, length_includes_head=True)
    
    offset_x = 0.15 if x_arrow >= 0 else -0.15
    offset_y = 0.12 if y_arrow >= 0 else -0.12
    ha = 'left' if x_arrow >= 0 else 'right'
    
    ax.text(x_arrow + offset_x, y_arrow + offset_y, nome_rotulo, 
            fontsize=8, fontweight='bold', ha=ha, va='center',
            bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.7, edgecolor='none'))

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)

ax.set_xlabel(f'Principal Component 1 (Latent Stress) - Explains {var_pc1_oficial:.2f}%', fontweight='bold')
ax.set_ylabel(f'Principal Component 2 (FX/Liquidity Risk) - Explains {var_pc2_oficial:.2f}%', fontweight='bold')

ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.savefig('biplot_pca.png', dpi=300, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 6. REGULARIZED MODELING (SPLIT 80/20 & EMPIRICAL RESULTS)
# ------------------------------------------------------------------------------
print("\n--- TRAINING REGULARIZED MODELS VIA GRIDSEARCH (80/20 SPLIT) ---")
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

alphas = np.logspace(-4, 4, 100)

grid_lasso = GridSearchCV(Lasso(max_iter=10000), {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error').fit(X_train_scaled, y_train)
grid_ridge = GridSearchCV(Ridge(), {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error').fit(X_train_scaled, y_train)
grid_en = GridSearchCV(ElasticNet(max_iter=10000), {'alpha': alphas, 'l1_ratio': [0.1, 0.5, 0.9]}, cv=5, scoring='neg_mean_squared_error').fit(X_train_scaled, y_train)

# Penalization Coefficients Table across Models
nomes_mercado_dinamicos = [f"{nome_limpo_map.get(v.split('_lag_')[0], v.split('_lag_')[0])} (t-{v.split('_lag_')[1]})" for v in features_dinamicas_col]
df_penalizacao = pd.DataFrame({
    'Market Indicator (Lagged)': nomes_mercado_dinamicos,
    'LASSO Weight (L1)': np.round(grid_lasso.best_estimator_.coef_, 4),
    'Ridge Weight (L2)': np.round(grid_ridge.best_estimator_.coef_, 4),
    'Elastic Net Weight': np.round(grid_en.best_estimator_.coef_, 4)
})
df_penalizacao.to_excel('regularization_weights_table.xlsx', index=False)

# Impact Bar Chart (LASSO Feature Importance)
df_plot = df_penalizacao.sort_values(by='LASSO Weight (L1)').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5.5))
colors = ['firebrick' if x < 0 else ('seagreen' if x > 0 else 'grey') for x in df_plot['LASSO Weight (L1)']]
bars = ax.barh(df_plot['Market Indicator (Lagged)'], df_plot['LASSO Weight (L1)'], color=colors, edgecolor='black', alpha=0.85)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.2)

for bar in bars:
    width = bar.get_width()
    if width < 0:
        ax.text(width - 0.08, bar.get_y() + bar.get_height()/2, f'{width:.4f}', va='center', ha='right', color='firebrick', fontweight='bold', fontsize=9)
    elif width > 0:
        ax.text(width + 0.08, bar.get_y() + bar.get_height()/2, f'{width:.4f}', va='center', ha='left', color='seagreen', fontweight='bold', fontsize=9)
    else:
        ax.text(0.05, bar.get_y() + bar.get_height()/2, '0.0000', va='center', ha='left', color='black', fontweight='bold', fontsize=9)

ax.set_xlim(df_plot['LASSO Weight (L1)'].min() - 0.6, df_plot['LASSO Weight (L1)'].max() + 0.6)
ax.set_xlabel('Statistical Weight and Directional Impact (Lagged Variables)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig('lasso_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# BIC Model Selection Validation Matrix
def calcular_bic(y_true, y_pred, num_features, n_samples):
    mse = mean_squared_error(y_true, y_pred)
    return n_samples * np.log(mse) + num_features * np.log(n_samples)

modelos_dict = {'LASSO': grid_lasso, 'Ridge': grid_ridge, 'Elastic Net': grid_en}
tabela_resultados = []

for nome, grid in modelos_dict.items():
    preds = grid.best_estimator_.predict(X_test_scaled)
    k = X_train_scaled.shape[1] if nome == 'Ridge' else np.sum(grid.best_estimator_.coef_ != 0)
    bic = calcular_bic(y_test, preds, k, len(y_test))
    tabela_resultados.append({'Model': nome, 'Active Coefs (k)': k, 'BIC Score': round(bic, 4)})

df_resultados = pd.DataFrame(tabela_resultados)
print("\n=== FINAL MODEL SELECTION MATRIX (BIC EVALUATION) ===")
print(df_resultados.to_string(index=False))

# ------------------------------------------------------------------------------
# 7. ISOLATED VIX HISTORICAL SERIES VIA LASSO (WITH CODACE SHADING)
# ------------------------------------------------------------------------------
modelo_vencedor = grid_lasso.best_estimator_
X_full_scaled_data = scaler_model.transform(X_full)
df_valid_dataset['ICE_Previsto_LASSO'] = modelo_vencedor.predict(X_full_scaled_data)

preds_lasso_full = df_valid_dataset['ICE_Previsto_LASSO']
vix_lasso_std = (preds_lasso_full - preds_lasso_full.mean()) / preds_lasso_full.std()
df_valid_dataset['VIX_Tupiniquim_LASSO'] = 100 - (vix_lasso_std * 10)

# Isolated Series Plot: VIX Tupiniquim (LASSO)
fig, ax = plt.subplots(figsize=(14, 5.2))
for inicio, fim in recessoes_codace:
    ax.axvspan(pd.to_datetime(inicio), pd.to_datetime(fim), color='lightgrey', alpha=0.6, 
               label='Official CODACE/FGV Recession' if inicio == "2014-03-01" else "")

ax.plot(df_valid_dataset['Data_Merge'], df_valid_dataset['VIX_Tupiniquim_LASSO'], 
        color='firebrick', linewidth=2.5, label='VIX Tupiniquim via LASSO (Base Mean = 100)')
ax.axhline(y=100, color='black', linestyle=':', linewidth=1.2, label='Historical Normalcy (Mean = 100)')
ax.set_xlabel('Years', fontweight='bold')
ax.set_ylabel('Index Points (Base Mean = 100)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(frameon=False, loc='best')
plt.tight_layout()
plt.savefig('vix_tupiniquim_lasso_base100.png', dpi=300, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 8. FINAL DATASET EXPORTATION
# ------------------------------------------------------------------------------
df_valid_dataset[['Data_Merge', 'VIX_Tupiniquim_PCA', 'VIX_Tupiniquim_LASSO']].rename(
    columns={'Data_Merge': 'Date', 'VIX_Tupiniquim_PCA': 'VIX_PCA', 'VIX_Tupiniquim_LASSO': 'VIX_LASSO'}
).to_excel('vix_tupiniquim_historical_series.xlsx', index=False)

print("\n--- PIPELINE SUCCESSFULLY CONVERTED AND EXECUTED! ---")